## Dataset analysis: `faculties_departments.csv`

Purpose: reference hierarchy (faculty → department → program). Used to validate/enrich `PROGRAM` values in other datasets.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()

df = pd.read_csv(DATA_DIR / "faculties_departments.csv")
df.shape

(91, 6)

In [2]:
df.head()

,faculty_id,faculty_name,department_id,department_name,program_id,program_name
0,1,Bishop Tucker School of Divinity and Theology,1,Divinity and Theology,6,Bachelor of Divinity
1,1,Bishop Tucker School of Divinity and Theology,1,Divinity and Theology,5,Doctor of Ministry
2,1,Bishop Tucker School of Divinity and Theology,1,Divinity and Theology,2,Master of Arts in Theology
3,1,Bishop Tucker School of Divinity and Theology,1,Divinity and Theology,3,Master of Divinity
4,1,Bishop Tucker School of Divinity and Theology,1,Divinity and Theology,1,Master of Theological Studies


In [3]:
df.isna().mean().sort_values(ascending=False)

faculty_id         0.0
faculty_name       0.0
department_id      0.0
department_name    0.0
program_id         0.0
program_name       0.0
dtype: float64

In [4]:
# Key uniqueness checks
key = ["faculty_id", "department_id", "program_id"]
df.duplicated(key).sum(), df[key].isna().any(axis=1).sum()

(np.int64(0), np.int64(0))

In [5]:
df.groupby(["faculty_name"]).agg(
    departments=("department_id", "nunique"),
    programs=("program_id", "nunique"),
).sort_values("programs", ascending=False).head(20)

,departments,programs
faculty_name,,
School of Education,2,17
School of Social Sciences,1,15
"Faculty of Engineering, Design & Technology",3,15
School of Business,5,13
Faculty of Agricultural Sciences,1,8
Bishop Tucker School of Divinity and Theology,1,6
"Faculty of Public Health, Nursing & Midwifery",1,6
School of Law,1,5
"School of Journalism, Media and Communication",1,4


## Advanced analytics

Focus: mapping programs to faculty/department and CGPA differences by faculty/department (via transcript join).

In [ ]:
from analysis_utils import basic_profile, missingness_report, group_summary

print(basic_profile(df))
missingness_report(df, top_n=20)

BasicProfile(rows=91, cols=6, dup_rows=0, null_cells=0)


,dtype,missing_rate,missing_count,nunique
faculty_id,int64,0.0,0,11
faculty_name,str,0.0,0,11
department_id,int64,0.0,0,18
department_name,str,0.0,0,18
program_id,int64,0.0,0,91
program_name,str,0.0,0,91


In [ ]:
# Join programs to transcript to see CGPA by faculty/department
trans_all = pd.concat([
    pd.read_csv(DATA_DIR / "student_transcript_list15.csv"),
    pd.read_csv(DATA_DIR / "student_transcript_list16.csv"),
], ignore_index=True)

trans_all["CGPA"] = pd.to_numeric(trans_all["CGPA"], errors="coerce")

merged = trans_all.merge(df, left_on="PROGRAM", right_on="program_name", how="left")

merged.groupby("faculty_name")["CGPA"].agg(["count","mean","median","std"]).sort_values("mean", ascending=False).head(20)

,count,mean,median,std
faculty_name,,,,
Faculty of Agricultural Sciences,104,3.435865,3.47,0.658521
School of Dentistry,4426,3.262289,3.29,0.643560
School of Business,4537,3.221962,3.25,0.647780
"Faculty of Public Health, Nursing & Midwifery",344,3.216744,3.25,0.663184
"Faculty of Engineering, Design & Technology",6029,3.189048,3.21,0.651119
School of Social Sciences,1465,3.187659,3.19,0.659338
Bishop Tucker School of Divinity and Theology,2177,3.122586,3.11,0.631599
School of Education,38,3.033158,3.16,0.693203
